In [26]:
import pandas as pd
import os
from pathlib import Path

In [27]:
def load_data(path: str) -> pd.DataFrame:
    """loads data from a csv file and returns a pandas DataFrame"""
    df = pd.read_csv(path)
    return df


def load_customers() -> pd.DataFrame:
    """loads customer data from the subject folder and returns a pandas DataFrame"""
    dfs = []
    base_path = Path.cwd()
    data_path = (base_path / "../subject/customer").resolve()

    if not data_path.exists():
        print(f"Directory not found: {data_path}")
        return

    for file in data_path.glob("*.csv"):
        dfs.append(pd.read_csv(file))
        
    contacts = pd.concat(dfs)
    contacts['event_time'] = pd.to_datetime(contacts['event_time'])
    return contacts


def load_items():
    """loads item data from the subject folder and returns a pandas DataFrame"""
    df = load_data("../subject/item/item.csv")
    return df


def merge_data(items, customers) -> pd.DataFrame:
    """merges the customer and item dataframes on the product_id column and returns a new DataFrame"""
    df = pd.merge(customers, items, on="product_id")
    return df


def remove_doubles(df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes rows where the same product, event, and user session 
    occur within 1 second of each other.
    """
    df = df.copy()
    df['event_time'] = pd.to_datetime(df['event_time'])
    
    sort_cols = ['user_id', 'product_id', 'event_type', 'event_time']
    df = df.sort_values(by=sort_cols)
    
    group_cols = ['user_id', 'product_id', 'event_type']
    time_diff = df.groupby(group_cols)['event_time'].diff()
    
    mask = (time_diff > pd.Timedelta(seconds=1)) | (time_diff.isna())
    
    return df[mask].reset_index(drop=True)

In [ ]:
customers = load_customers()
customers = remove_doubles(customers)
items = load_items()
customers = merge_data(items, customers)

In [ ]:
frequency = customers[customers['event_type'] == "purchase"]
frequency = frequency.groupby("user_id").size().reset_index(name="purchase_count")
frequency.head()

,user_id,purchase_count
0,9794320,12
1,10079204,5
2,10280338,41
3,12055855,10
4,12936739,5


In [ ]:
monetary = customers[customers['event_type'] == "purchase"]
monetary = monetary.groupby("user_id")["price"].sum().reset_index(name="total_spent")
monetary.head()

,user_id,total_spent
0,9794320,38.04
1,10079204,65.18
2,10280338,186.70
3,12055855,42.05
4,12936739,86.75


In [ ]:
data = pd.merge(frequency, monetary, on="user_id")
data.head()


,user_id,purchase_count,total_spent
0,9794320,12,38.04
1,10079204,5,65.18
2,10280338,41,186.70
3,12055855,10,42.05
4,12936739,5,86.75


In [ ]:
last_purchase_date = customers[customers['event_type'] == "purchase"].groupby("user_id")["event_time"].max().reset_index(name="last_purchase_date")
latest_date = last_purchase_date['last_purchase_date'].max()
data['days_since_last'] = (latest_date - last_purchase_date['last_purchase_date']).dt.days
data.head()

,user_id,purchase_count,total_spent,last_purchase_date_x,days_since_last,last_purchase_date_y,last_purchase_date
0,9794320,12,38.04,2022-11-25 05:07:13+00:00,67,2022-11-25 05:07:13+00:00,2022-11-25 05:07:13+00:00
1,10079204,5,65.18,2022-11-06 10:43:30+00:00,86,2022-11-06 10:43:30+00:00,2022-11-06 10:43:30+00:00
2,10280338,41,186.70,2023-01-12 22:54:37+00:00,19,2023-01-12 22:54:37+00:00,2023-01-12 22:54:37+00:00
3,12055855,10,42.05,2022-12-20 12:05:29+00:00,42,2022-12-20 12:05:29+00:00,2022-12-20 12:05:29+00:00
4,12936739,5,86.75,2023-01-17 07:51:19+00:00,14,2023-01-17 07:51:19+00:00,2023-01-17 07:51:19+00:00
